In [ ]:
import mcstasscript as ms
import make_QENS_instrument

In [ ]:
instrument = make_QENS_instrument.make()

In [ ]:
instrument.set_parameters(sample_distance=150, energy_width_ueV=5, sample_choice='"Elastic"', n_pulses=1)
instrument.settings(ncount=1e7, mpi="auto", suppress_output=True, NeXus=True, output_path="first_run")

data = instrument.backengine()


In [ ]:
ms.make_sub_plot(data, figsize=(9.5, 5))

In [ ]:
make_QENS_instrument.add_chopper_code(instrument)

chopper = instrument.add_component("chopper", "DiskChopper", after="source")
chopper.set_parameters(
    yheight=0.05,
    radius=0.7,
    nu="chopper_frequency",
    nslit=1.0,
    delay="chopper_delay",
    theta_0="chopper_theta",
)


chopper.set_AT([0, 0, "chopper_distance"], RELATIVE="source")

In [ ]:
instrument.settings(output_path="QENS_elastic")
instrument.set_parameters(energy_width_ueV=5, sample_choice='"Elastic"', n_pulses=1, frequency_multiplier=10)

data_improved = instrument.backengine()

In [ ]:
ms.make_sub_plot(data_improved, figsize=(9.5, 5))

In [ ]:
instrument.settings(output_path="QENS_known_quasi_elastic", ncount=5E7)
instrument.set_parameters(energy_width_ueV=100, sample_choice='"Known_quasi-elastic"',
                          gamma_ueV=3, frequency_multiplier=10)

data_known = instrument.backengine()

In [ ]:
ms.make_sub_plot(data_known, figsize=(9.5, 5), log=True)

In [ ]:
instrument.show_parameters()

In [ ]:
# load
# SPDX-License-Identifier: BSD-3-Clause
# Copyright (c) 2023 Scipp contributors (https://github.com/scipp)

import os
import pandas as pd
import scipp as sc
import scippnexus as sx
from typing import Tuple
import warnings


def load_nexus(path: str, index) -> sc.DataArray:
    """
    Load a SANS nexus file and return a scipp DataArray with the data.
    """
    fname = os.path.join(path, "mccode.h5")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with sx.File(fname) as f:
            dg = f[...]
    events = sc.collapse(
        dg["entry1"]["data"][f"detector_signal_event_{index}_dat"].data, keep="dim_0"
    )
    meta = dg["entry1"]["simulation"]["Param"]
    columns = ["p", "x", "y", "n", "id", "t"]
    events = {
        c: v.rename_dims(dim_0="event").copy() for c, v in zip(columns, events.values())
    }
    return events, meta


def _load_header(fname: str, comment: str = "#") -> dict:
    lines = []
    maxlines = 100
    with open(fname, "r") as f:
        for _ in range(maxlines):
            line = f.readline()
            if line.startswith(comment):
                lines.append(line.lstrip(f" {comment}").strip())
            else:
                break
    header = {}
    for l in lines:
        if l.startswith("Param"):
            key, value = l.split(":")[1].split("=")
            header[key.strip()] = value.strip()
        else:
            pieces = l.split(":")
            if len(pieces) == 2:
                value = pieces[1].strip()
            else:
                value = pieces[1:]
            header[pieces[0].strip()] = value
    return header


def load_ascii(
    filename: str,
) -> Tuple[sc.DataArray, dict]:
    meta = _load_header(fname=filename)

    ds = sc.compat.from_pandas(
        pd.read_csv(
            filename,
            delimiter=" ",
            comment="#",
            names=["p", "x", "y", "n", "id", "t"],
            index_col=False,
        )
    )
    events = {key: c.data.rename_dims(row="event") for key, c in ds.items()}
    return events, meta


In [ ]:
# qens_utils modified

# SPDX-License-Identifier: BSD-3-Clause
# Copyright (c) 2023 Scipp contributors (https://github.com/scipp)

import os
import scipp as sc


DETECTOR_OFFSET = 0.25 * sc.units.m


def analyzer_info(params: sc.DataGroup, angle) -> sc.DataGroup:
    # This assumes that the analyzer is in the source-sample plane,
    # rotated by `angle` about the y-axis around the sample
    # and at `distance` from the sample.

    distance = sc.scalar(float(params["sample_analyzer_distance"]), unit="m")
    analyzer_position = sc.spatial.as_vectors(
        sc.sin(angle) * distance,
        sc.scalar(0.0, unit="m"),
        sc.cos(angle) * distance,
    )

    # The analyzer is tilted by this angle such that neutron are
    # reflected by `2*analyzer_angle` to the detector.
    analyzer_angle = sc.atan2(y=DETECTOR_OFFSET, x=distance) / 2

    return sc.DataGroup(
        {
            # Si (111) as Miracles: Q = 2*pi/3.135
            "analyzer_dspacing": sc.scalar(3.135, unit="angstrom"),
            "analyzer_position": analyzer_position,
            "analyzer_angle": analyzer_angle,
            "two_theta": angle,
        }
    )


def correct_tof(tof):
    # The instrument focuses on the center of the pulse at 2.86/2 ms.
    # Shift the time such that tof is the time since the neutron were emitted.
    return tof - sc.scalar(0.5 * 2.86, unit="ms")


def load_qens(path: str, index) -> sc.DataArray:
    """
    Load a QENS nexus file for the summer school QENS experiment.

    Parameters
    ----------
    path
        Path to the directory containing the simulation results.
    """
    ascii_file = os.path.join(path, f"detector_signal_event_{index}.dat")
    if os.path.exists(ascii_file):
        events, meta = load_ascii(filename=ascii_file)
    else:
        events, meta = load_nexus(path=path, index=index)

    weights = events.pop("p")
    weights.unit = "counts"
    weights *= float(meta["integration_time"])
    da = sc.DataArray(data=weights, coords=events)

    # Add variances
    # (See https://www.mcstas.org/documentation/manual/mcstas-3.5.27-manual.pdf,
    # section 2.2.1)
    da.variances = da.values**2

    # todo
    # This position needs to be update slightly, distance from analyzer back to detector is smaller
    # The analyzer to detector distance is available as a parameter: analyzer_detector_distance

    # Need to match McStas simulation
    analyzer_directions = [20, 50, 80, 110, 140]
    angle = sc.scalar(float(analyzer_directions[index]), unit="deg")


    # Position of detectors a bit different now
    distance = sc.scalar(float(meta["sample_analyzer_distance"]) - float(meta["analyzer_detector_distance"]), unit="m")
    
    da.coords["y"].unit = "m"
    # The event positions are in the detector coordinate system.
    # Translate by the detector offset to get the lab system.
    da.coords["y"] += DETECTOR_OFFSET
    da.coords["x"].unit = "m"
    da.coords["x"] += sc.sin(angle) * distance
    da.coords["z"] = sc.zeros_like(da.coords["y"])
    da.coords["z"] += sc.cos(angle) * distance
    
    
    da.coords["position"] = sc.spatial.as_vectors(
        da.coords["x"].to(dtype=float), da.coords["y"], da.coords["z"]
    )
    
    da.coords["tof"] = da.coords.pop("t")
    da.coords["tof"].unit = "s"
    da.coords["tof"] = correct_tof(da.coords["tof"].to(unit="ms"))

    da.coords["sample_position"] = sc.vector([0.0, 0.0, 0.0], unit="m")
    da.coords["source_position"] = sc.vector(
        [0.0, 0.0, -float(meta["sample_distance"])], unit="m"
    )

    da.coords.update(analyzer_info(meta, angle=angle))

    return da


In [ ]:
folder = data_known[0].original_data_location

events = load_qens(folder, 1)

In [ ]:
events.hist(tof=100, y=100).plot() # See glitch is present

In [ ]:
y = events.coords["y"]
mask_regions = sc.array(
    dims=["y"],
    values=[0.21, 0.26, 0.27, 0.29],
    unit=y.unit,
)
binned_for_mask = events.bin(y=mask_regions)
mask = sc.array(dims=["y"], values=[False, True, False])
binned_for_mask.masks["bad_timing"] = mask
binned_for_mask

In [ ]:
import numpy as np

def backscattered_l2(position, sample_position, analyzer_position):
    """
    Compute the length of the secondary flight path for backscattering off an analyzer.
    """
    return sc.norm(position - analyzer_position) + sc.norm(
        analyzer_position - sample_position
    )


def wavelength_from_analyzer(analyzer_dspacing, analyzer_angle):
    """
    Compute the neutron wavelength after scattering from the analyzer's d-spacing.

    Assuming Bragg scattering in the analyzer, the wavelength is
        wavelength = 2 * d * sin(theta)

    Where
        d is the analyzer's d-spacing,
        theta is the scattering angle or equivalently, the tilt of the analyzer
              w.r.t. to the sample-analyzer axis.
    """
    # 2*theta is the angle between transmitted and scattered beam.
    return (
        2
        * analyzer_dspacing
        * sc.sin(sc.scalar(np.pi / 2, unit="rad") - analyzer_angle.to(unit="rad"))
    )

In [ ]:
from scippneutron.conversion.graph.beamline import beamline
from scippneutron.conversion.tof import energy_transfer_indirect_from_tof

graph = {
    **beamline(scatter=True),
    "energy_transfer": energy_transfer_indirect_from_tof,
    # Replace L2 with our own implementation.
    "L2": backscattered_l2,
    # Insert a new function for the wavelength.
    "final_wavelength": wavelength_from_analyzer,
}
# Optional: remove unused functions in order to clean up the image below.
del graph["two_theta"]
del graph["scattered_beam"]
del graph["Ltotal"]
sc.show_graph(graph, simplified=True)

In [ ]:
def final_energy(final_wavelength):
    """
    Compute the neutron energy after scattering.

    Uses
        final_energy = mn / 2 * final_speed**2
        final_speed = 2 * pi * hbar / mn / final_wavelength

    Where
        mn is the neutron mass,
        final_wavelength is the wavelength after scattering,
        final_speed is the speed after scattering.
    """
    return sc.to_unit(
        sc.constants.h**2 / 2 / sc.constants.neutron_mass / (final_wavelength**2),
        "meV",
    )


graph["final_energy"] = final_energy
sc.show_graph(graph, simplified=True)

In [ ]:
in_energy_transfer = binned_for_mask.transform_coords("energy_transfer", graph=graph)
in_energy_transfer

In [ ]:
in_energy_transfer.bins.concat().hist(energy_transfer=100).plot()

In [ ]:
in_energy_transfer.bins.concat().hist(tof=100, y=100).plot() # See glitch is masked

In [ ]:
def process_sample(raw_events, bin_width=0.001 * sc.Unit('meV'), apply_mask=False):

    if apply_mask:
        binned_for_mask = raw_events.bin(y=mask_regions)
        binned_for_mask.masks["bad_timing"] = mask

        in_energy_transfer = binned_for_mask.transform_coords(
            "energy_transfer", graph=graph
        )
    
        hist = in_energy_transfer.bins.concat().hist(energy_transfer=bin_width)
    else:
        in_energy_transfer = raw_events.transform_coords(
            "energy_transfer", graph=graph
        )
        hist = in_energy_transfer.hist(energy_transfer=bin_width)

    two_theta = in_energy_transfer.coords["two_theta"]

    return hist, two_theta

In [ ]:
results = {}
for index in range(5):
    events = load_qens(folder, index)
    apply_mask = index == 1
    # Remove the file path prefix from the folder name
    hist, two_theta = process_sample(events, apply_mask=apply_mask) 
    
    print(two_theta)
    results[two_theta.value] = hist

In [ ]:
hists = list(results.values())

hists[0].plot() / hists[1].plot() / hists[2].plot() / hists[3].plot() / hists[4].plot()

In [ ]:
in_energy_transfer.coords["two_theta"]

In [ ]:
from scippneutron.io import save_xye

for name, result in results.items():
    # The simple file format does not support bin-edge coordinates.
    # So we convert to bin-centers first.
    data = result.copy()
    data.coords["energy_transfer"] = sc.midpoints(data.coords["energy_transfer"])

    save_xye(f"energy_transfer_{name}.dat", data)